# 📑 Document Version Comparator (Gemini Edition)

Compare **two or more versions** of a document (reports, assignments, policy files, contracts, ...)
and automatically generate a structured **difference report** showing:

- ➕ **Additions** — new content introduced in a later version
- ➖ **Deletions** — content removed from an earlier version
- ✏️ **Modifications** — blocks that were edited (with old vs. new text)
- 🧠 **Gemini-powered change summary** — a plain-English explanation of what changed and why it might matter
  (e.g. wording tweak vs. factual change vs. policy change), useful for review before submission/approval

**Supported input formats:** `.txt`, `.docx`, `.pdf`

**Powered by:** [Google Gemini API](https://ai.google.dev/) for semantic classification & summarization of detected diffs.
The raw diffing itself (matching text blocks across versions) is done locally with Python's `difflib` —
no document content needs to leave your machine unless you choose to run the Gemini summary step.

---
### How to use
1. Run the **Setup** cells once.
2. Add your `GEMINI_API_KEY` in the **Configuration** cell.
3. List your document version file paths, in chronological order, in **Section 3**.
4. Run **Section 4** to extract text and split each version into comparable blocks.
5. Run **Section 5** to compute block-level diffs between consecutive versions.
6. (Optional) Run **Section 6** to get a Gemini-generated plain-English summary of the changes.
7. Run **Section 7** to export a structured difference report as Markdown + Word (`.docx`).


## 1. Setup — Install Dependencies

In [ ]:
import sys

# Install required packages (run once)
# - python-docx: read/write .docx files
# - pdfplumber: extract text from .pdf files
# - google-generativeai: Gemini API client for semantic change summaries
# - python-docx is also used to generate the final report

!{sys.executable} -m pip install -q python-docx pdfplumber google-generativeai
print("Done.")


## 2. Configuration

Set your Google Gemini API key (get one at https://aistudio.google.com/app/apikey).
You can also set it as an environment variable `GEMINI_API_KEY` instead of pasting it here.

In [ ]:
import os
import re
import json
import difflib
from datetime import datetime
import google.generativeai as genai

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # <-- paste your key between quotes if needed

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("Gemini client configured and ready.")
else:
    print("⚠️ No API key set yet — Section 6 (AI change summary) will need this before it can run.")
    print("   Sections 1-5 and 7 work fine without it (pure text diffing + report export).")

# Check this against Google's current model list at https://ai.google.dev/gemini-api/docs/models
# if you hit a 404 / not-found error when calling the API.
GEMINI_MODEL = "gemini-1.5-flash"

# Minimum similarity ratio (0-1) for two blocks to be considered a "modification" of each other
# rather than an unrelated deletion + addition pair. Lower = more aggressive matching.
MODIFICATION_SIMILARITY_THRESHOLD = 0.55


## 3. Input — Document Versions

List the file paths of each version **in chronological order** (oldest → newest).
Upload the files to Jupyter first (e.g. into the same folder as this notebook).

You need **at least 2** versions. You can list 3+ to compare a whole revision chain —
each consecutive pair (v1→v2, v2→v3, ...) will get its own diff report.

In [ ]:
DOCUMENT_VERSIONS = [
    "report_v1.docx",   # oldest
    "report_v2.docx",
    # "report_v3.docx",  # add as many as needed, in order
]

VERSION_LABELS = [f"Version {i+1}" for i in range(len(DOCUMENT_VERSIONS))]
# Optionally customize labels, e.g.:
# VERSION_LABELS = ["Draft", "Reviewer Edits", "Final Submission"]

print(f"{len(DOCUMENT_VERSIONS)} version(s) listed:")
for path, label in zip(DOCUMENT_VERSIONS, VERSION_LABELS):
    exists = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  [{label}] {path}  {exists}")

if len(DOCUMENT_VERSIONS) < 2:
    print("\n⚠️ Add at least 2 document versions above before continuing.")


## 4. Extract Text & Split into Blocks

Each document is read and broken into **blocks** (roughly paragraph-level chunks). Blocks are the unit
the diff engine compares — this gives more meaningful results than line-by-line or character-by-character
diffing, since edits to a sentence within a paragraph still register as one "modified block" rather than
a scramble of single-line changes.

In [ ]:
def extract_text(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    elif ext == ".docx":
        from docx import Document
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    elif ext == ".pdf":
        import pdfplumber
        text_parts = []
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text_parts.append(page.extract_text() or "")
        return "\n".join(text_parts)
    else:
        raise ValueError(f"Unsupported file type: {ext} (supported: .txt, .docx, .pdf)")

def split_into_blocks(text):
    """Split text into paragraph-level blocks, dropping empty/whitespace-only blocks."""
    raw_blocks = re.split(r"\n\s*\n|\n(?=\s*[A-Z0-9])", text) if "\n\n" not in text else text.split("\n\n")
    # Fallback simpler approach: split on blank lines, then further split very long blocks by single newlines
    blocks = [b.strip() for b in text.split("\n\n")]
    blocks = [b for b in blocks if b]
    if len(blocks) <= 1 and "\n" in text:
        # Document had no blank-line paragraph breaks (common after .docx extraction) — fall back to line splits
        blocks = [b.strip() for b in text.split("\n") if b.strip()]
    return blocks

version_texts = []
version_blocks = []

for path in DOCUMENT_VERSIONS:
    text = extract_text(path)
    blocks = split_into_blocks(text)
    version_texts.append(text)
    version_blocks.append(blocks)

for label, blocks in zip(VERSION_LABELS, version_blocks):
    print(f"[{label}] {len(blocks)} block(s) extracted")


## 5. Block-by-Block Diff Engine

For each consecutive pair of versions, this aligns blocks using `difflib.SequenceMatcher` and classifies
each change as an **addition**, **deletion**, or **modification** (a deleted block and an added block that
are similar enough to be treated as an edit of one another, rather than two unrelated changes).

In [ ]:
def diff_blocks(old_blocks, new_blocks, threshold=MODIFICATION_SIMILARITY_THRESHOLD):
    """Returns a list of change dicts: {type, old_text, new_text, old_index, new_index, similarity}"""
    sm = difflib.SequenceMatcher(a=old_blocks, b=new_blocks, autojunk=False)
    changes = []

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        elif tag == "insert":
            for j in range(j1, j2):
                changes.append({"type": "addition", "old_text": None, "new_text": new_blocks[j],
                                 "old_index": None, "new_index": j, "similarity": None})
        elif tag == "delete":
            for i in range(i1, i2):
                changes.append({"type": "deletion", "old_text": old_blocks[i], "new_text": None,
                                 "old_index": i, "new_index": None, "similarity": None})
        elif tag == "replace":
            old_chunk = old_blocks[i1:i2]
            new_chunk = new_blocks[j1:j2]
            # Try to pair up replaced blocks by similarity (treat as modifications where similar enough)
            unmatched_old = list(range(len(old_chunk)))
            unmatched_new = list(range(len(new_chunk)))
            pairs = []
            for oi in list(unmatched_old):
                best_j, best_score = None, 0.0
                for nj in unmatched_new:
                    score = difflib.SequenceMatcher(a=old_chunk[oi], b=new_chunk[nj], autojunk=False).ratio()
                    if score > best_score:
                        best_score, best_j = score, nj
                if best_j is not None and best_score >= threshold:
                    pairs.append((oi, best_j, best_score))
                    unmatched_old.remove(oi)
                    unmatched_new.remove(best_j)
            for oi, nj, score in pairs:
                changes.append({"type": "modification", "old_text": old_chunk[oi], "new_text": new_chunk[nj],
                                 "old_index": i1 + oi, "new_index": j1 + nj, "similarity": round(score, 3)})
            for oi in unmatched_old:
                changes.append({"type": "deletion", "old_text": old_chunk[oi], "new_text": None,
                                 "old_index": i1 + oi, "new_index": None, "similarity": None})
            for nj in unmatched_new:
                changes.append({"type": "addition", "old_text": None, "new_text": new_chunk[nj],
                                 "old_index": None, "new_index": j1 + nj, "similarity": None})
    return changes

# Compute diffs for every consecutive pair of versions
pairwise_diffs = []  # list of {"from": label, "to": label, "changes": [...]}

for k in range(len(version_blocks) - 1):
    changes = diff_blocks(version_blocks[k], version_blocks[k + 1])
    pairwise_diffs.append({
        "from": VERSION_LABELS[k],
        "to": VERSION_LABELS[k + 1],
        "changes": changes,
    })
    n_add = sum(1 for c in changes if c["type"] == "addition")
    n_del = sum(1 for c in changes if c["type"] == "deletion")
    n_mod = sum(1 for c in changes if c["type"] == "modification")
    print(f"{VERSION_LABELS[k]} → {VERSION_LABELS[k+1]}: "
          f"{n_add} addition(s), {n_del} deletion(s), {n_mod} modification(s)")


## 6. AI Change Summary (Gemini)

Sends the detected changes (not the entire documents) to Gemini to generate a plain-English review summary:
overall nature of the edits, and a classification of each change as a **wording/style tweak**,
a **factual/content change**, a **structural change**, or a **policy/numeric change** — useful for spotting
which edits actually need a reviewer's attention versus which are cosmetic.

In [ ]:
SUMMARY_PROMPT = """You are an expert document reviewer. Below is a list of detected changes between
two versions of a document ("{from_label}" -> "{to_label}"). Each change is one of: addition, deletion,
or modification (showing old_text and new_text).

Analyze these changes and return a JSON object with EXACTLY these keys:
{{
  "overall_summary": "2-4 sentence plain-English summary of what changed overall between these two versions",
  "classified_changes": [
    {{
      "change_index": 0,
      "category": "wording|factual|structural|policy_or_numeric|formatting",
      "significance": "low|medium|high",
      "explanation": "one sentence on what this specific change does and why it matters (or doesn't)"
    }}
  ],
  "review_recommendation": "1-2 sentence recommendation for what a reviewer should focus on before approving this version"
}}

change_index must match the position of each change in the list below (0-indexed). Return ONLY valid JSON.

CHANGES:
{changes_json}"""

def summarize_changes_with_gemini(from_label, to_label, changes):
    if not GEMINI_API_KEY:
        raise RuntimeError("Gemini API key not configured. Set GEMINI_API_KEY in Section 2.")
    if not changes:
        return {"overall_summary": "No differences detected between these two versions.",
                "classified_changes": [], "review_recommendation": "No action needed."}

    # Keep payload lean: trim very long block text before sending
    trimmed = [
        {"type": c["type"],
         "old_text": (c["old_text"][:600] + "...") if c["old_text"] and len(c["old_text"]) > 600 else c["old_text"],
         "new_text": (c["new_text"][:600] + "...") if c["new_text"] and len(c["new_text"]) > 600 else c["new_text"]}
        for c in changes
    ]
    prompt = SUMMARY_PROMPT.format(from_label=from_label, to_label=to_label,
                                    changes_json=json.dumps(trimmed, indent=2))

    model = genai.GenerativeModel(model_name=GEMINI_MODEL,
                                   generation_config={"response_mime_type": "application/json"})
    response = model.generate_content(prompt)
    try:
        return json.loads(response.text)
    except json.JSONDecodeError:
        print("Failed to decode JSON from Gemini response. Raw response was:")
        print(response.text)
        return {"overall_summary": "", "classified_changes": [], "review_recommendation": ""}

ai_summaries = []  # parallel to pairwise_diffs

if GEMINI_API_KEY:
    for pair in pairwise_diffs:
        print(f"Summarizing changes: {pair['from']} → {pair['to']} ...")
        summary = summarize_changes_with_gemini(pair["from"], pair["to"], pair["changes"])
        ai_summaries.append(summary)
    print("✅ AI summaries generated.")
else:
    ai_summaries = [None] * len(pairwise_diffs)
    print("⚠️ Skipped — set GEMINI_API_KEY in Section 2 to enable AI summaries.")


## 7. Generate Structured Difference Report

In [ ]:
def truncate(s, n=300):
    if s is None:
        return ""
    return s if len(s) <= n else s[:n] + "..."

def render_report_markdown(pair, ai_summary):
    out = [f"# 📑 Difference Report: {pair['from']} → {pair['to']}", ""]
    out.append(f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_\n")

    if ai_summary and ai_summary.get("overall_summary"):
        out.append("## 🧠 Overall Summary")
        out.append(ai_summary["overall_summary"] + "\n")

    changes = pair["changes"]
    classified = {}
    if ai_summary and ai_summary.get("classified_changes"):
        classified = {c["change_index"]: c for c in ai_summary["classified_changes"]}

    n_add = sum(1 for c in changes if c["type"] == "addition")
    n_del = sum(1 for c in changes if c["type"] == "deletion")
    n_mod = sum(1 for c in changes if c["type"] == "modification")
    out.append(f"## Summary of Changes\n- ➕ Additions: {n_add}\n- ➖ Deletions: {n_del}\n- ✏️ Modifications: {n_mod}\n")

    out.append("## Detailed Changes\n")
    if not changes:
        out.append("No differences detected between these versions.")
    for idx, c in enumerate(changes):
        icon = {"addition": "➕", "deletion": "➖", "modification": "✏️"}[c["type"]]
        out.append(f"### {icon} Change {idx+1} — {c['type'].capitalize()}")
        meta = classified.get(idx)
        if meta:
            out.append(f"_Category: **{meta.get('category','')}** | Significance: **{meta.get('significance','')}**_")
        if c["type"] == "addition":
            out.append(f"**Added:** {truncate(c['new_text'])}")
        elif c["type"] == "deletion":
            out.append(f"**Removed:** {truncate(c['old_text'])}")
        elif c["type"] == "modification":
            out.append(f"**Before:** {truncate(c['old_text'])}")
            out.append(f"**After:** {truncate(c['new_text'])}")
            if c.get("similarity") is not None:
                out.append(f"_Similarity to previous version: {c['similarity']*100:.0f}%_")
        if meta and meta.get("explanation"):
            out.append(f"> {meta['explanation']}")
        out.append("")

    if ai_summary and ai_summary.get("review_recommendation"):
        out.append("## ✅ Review Recommendation")
        out.append(ai_summary["review_recommendation"])

    return "\n".join(out)

from IPython.display import Markdown, display

report_markdowns = []
for pair, ai_summary in zip(pairwise_diffs, ai_summaries):
    report_md = render_report_markdown(pair, ai_summary)
    report_markdowns.append(report_md)
    display(Markdown(report_md))
    display(Markdown("---"))


## 8. Export Reports (Markdown + Word)

In [ ]:
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", s)[:60]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
saved_files = []

for pair, report_md in zip(pairwise_diffs, report_markdowns):
    fname = f"diff_{safe_name(pair['from'])}_to_{safe_name(pair['to'])}_{timestamp}.md"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(report_md)
    saved_files.append(fname)

# Also export each report as a formatted Word document
from docx import Document as DocxDocument
from docx.shared import RGBColor

TYPE_COLORS = {
    "addition": RGBColor(0x1B, 0x7A, 0x1B),
    "deletion": RGBColor(0xB0, 0x2A, 0x2A),
    "modification": RGBColor(0xB8, 0x86, 0x0B),
}

for pair, ai_summary in zip(pairwise_diffs, ai_summaries):
    doc = DocxDocument()
    doc.add_heading(f"Difference Report: {pair['from']} \u2192 {pair['to']}", level=0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

    if ai_summary and ai_summary.get("overall_summary"):
        doc.add_heading("Overall Summary", level=1)
        doc.add_paragraph(ai_summary["overall_summary"])

    changes = pair["changes"]
    classified = {c["change_index"]: c for c in ai_summary.get("classified_changes", [])} if ai_summary else {}

    n_add = sum(1 for c in changes if c["type"] == "addition")
    n_del = sum(1 for c in changes if c["type"] == "deletion")
    n_mod = sum(1 for c in changes if c["type"] == "modification")
    doc.add_heading("Summary of Changes", level=1)
    doc.add_paragraph(f"Additions: {n_add}   |   Deletions: {n_del}   |   Modifications: {n_mod}")

    doc.add_heading("Detailed Changes", level=1)
    if not changes:
        doc.add_paragraph("No differences detected between these versions.")
    for idx, c in enumerate(changes):
        p = doc.add_heading(f"Change {idx+1} — {c['type'].capitalize()}", level=2)
        for run in p.runs:
            run.font.color.rgb = TYPE_COLORS.get(c["type"])
        meta = classified.get(idx)
        if meta:
            doc.add_paragraph(f"Category: {meta.get('category','')} | Significance: {meta.get('significance','')}")
        if c["type"] == "addition":
            doc.add_paragraph(f"Added: {truncate(c['new_text'], 1000)}")
        elif c["type"] == "deletion":
            doc.add_paragraph(f"Removed: {truncate(c['old_text'], 1000)}")
        elif c["type"] == "modification":
            doc.add_paragraph(f"Before: {truncate(c['old_text'], 1000)}")
            doc.add_paragraph(f"After: {truncate(c['new_text'], 1000)}")
        if meta and meta.get("explanation"):
            doc.add_paragraph(meta["explanation"]).italic = True

    if ai_summary and ai_summary.get("review_recommendation"):
        doc.add_heading("Review Recommendation", level=1)
        doc.add_paragraph(ai_summary["review_recommendation"])

    docx_name = f"diff_{safe_name(pair['from'])}_to_{safe_name(pair['to'])}_{timestamp}.docx"
    doc.save(docx_name)
    saved_files.append(docx_name)

print("✅ Saved files:")
for f in saved_files:
    print(" -", f)


## 9. Tips

- **Block granularity:** the diff engine compares paragraph-level blocks by default. If your documents use
  single line breaks instead of blank lines between paragraphs (common after `.docx` extraction), the
  fallback in `split_into_blocks` switches to line-level splitting automatically — check the block counts
  printed in Section 4 to make sure they look reasonable for your document.
- **Tuning sensitivity:** raise `MODIFICATION_SIMILARITY_THRESHOLD` (Section 2) if unrelated blocks are
  being incorrectly paired as "modifications"; lower it if genuinely edited paragraphs are showing up as a
  separate deletion + addition instead of one modification.
- **More than 2 versions:** list all versions in Section 3 in order — the system diffs each consecutive
  pair (v1→v2, v2→v3, ...) and produces a separate report for each, so you can track the evolution
  step-by-step rather than only the first-vs-last comparison.
- **Skipping AI summary:** Sections 1–5, 7, and 8 work with no API key at all — you'll still get full
  addition/deletion/modification detection and exported reports; only the plain-English categorization
  and review recommendation require Gemini.
- **Large documents:** Gemini handles long change lists well, but very large documents may produce a long
  list of changes — consider running per-section comparisons if a single document spans hundreds of pages.
